# Benchmark Results EDA

### Table of contents

1. [Data loading](#data-loading)
2. [Overview](#overview)
3. [F1 performance](#f1-performance)
   - [Overall accuracy](#overall-accuracy)
   - [Exact vs. fuzzy gap](#exact-vs-fuzzy-gap)
   - [Per-field breakdown](#per-field-breakdown)
4. [Score distributions](#score-distributions)
5. [Efficiency](#efficiency)
   - [Cost vs. accuracy](#cost-vs-accuracy)
   - [Latency](#latency)
6. [Conclusions](#conclusions)

## Introduction

This notebook explores the benchmark results across 12 experiments — three providers (Claude, Gemini, GPT), two model tiers (lite, standard), and two extraction strategies (agentic multimodal, single-pass text). Each experiment was run on 83 NDA documents from the Kleister-NDA dev split.


In [4]:
from pathlib import Path

import altair as alt
import polars as pl


## Data loading


In [5]:
data_path = Path.cwd() / "data" / "results"

agg = pl.read_csv(data_path / "benchmark-aggregated.csv")
per_doc = pl.read_csv(data_path / "benchmark-per-document.csv")

print(
    f"Aggregated: {agg.shape[0]} experiments × {agg.shape[1]} columns\n"
    f"Per-document: {per_doc.shape[0]} rows × {per_doc.shape[1]} columns"
)


Aggregated: 12 experiments × 26 columns
Per-document: 996 rows × 19 columns


## Overview


In [6]:
print(
    f"Providers:  {agg['provider'].unique().sort().to_list()}\n"
    f"Tiers:      {agg['tier'].unique().sort().to_list()}\n"
    f"Strategies: {agg['strategy'].unique().sort().to_list()}\n"
    f"Documents per experiment: {agg['n_docs'][0]}\n"
    f"\nExact F1 — min: {agg['exact_f1_mean'].min():.3f},  max: {agg['exact_f1_mean'].max():.3f}"
)
display(
    agg.select(["provider", "tier", "strategy", "exact_f1_mean", "fuzzy_f1_mean", "latency_mean", "cost_mean"])
    .sort(["tier", "provider", "strategy"])
)


Providers:  ['claude', 'gemini', 'gpt']
Tiers:      ['lite', 'standard']
Strategies: ['agentic', 'single_pass']
Documents per experiment: 83

Exact F1 — min: 0.672,  max: 0.918


provider,tier,strategy,exact_f1_mean,fuzzy_f1_mean,latency_mean,cost_mean
str,str,str,f64,f64,f64,f64
"""claude""","""lite""","""agentic""",0.745563,0.761806,31.012886,0.010601
"""claude""","""lite""","""single_pass""",0.871918,0.888412,16.891371,0.006079
"""gemini""","""lite""","""agentic""",0.840046,0.86156,5.554535,0.002639
"""gemini""","""lite""","""single_pass""",0.89567,0.91116,1.413686,0.001075
"""gpt""","""lite""","""agentic""",0.671516,0.682187,11.358014,0.005846
…,…,…,…,…,…,…
"""claude""","""standard""","""single_pass""",0.856813,0.87381,33.339101,0.020893
"""gemini""","""standard""","""agentic""",0.918359,0.936359,14.567474,0.010635
"""gemini""","""standard""","""single_pass""",0.915347,0.936359,9.789317,0.00748


## F1 performance

### Overall accuracy


In [7]:
def plot_overall_f1(df: pl.DataFrame) -> alt.Chart:
    return (
        alt.Chart(df.to_pandas())
        .mark_bar()
        .encode(
            x=alt.X("provider:N", title=None),
            y=alt.Y(
                "exact_f1_mean:Q",
                title="Exact F1",
                scale=alt.Scale(domain=[0.6, 1.0]),
            ),
            color=alt.Color("strategy:N", title="Strategy"),
            column=alt.Column("tier:N", title="Tier"),
            tooltip=[
                alt.Tooltip("provider:N"),
                alt.Tooltip("tier:N"),
                alt.Tooltip("strategy:N"),
                alt.Tooltip("exact_f1_mean:Q", format=".3f", title="Exact F1"),
                alt.Tooltip("fuzzy_f1_mean:Q", format=".3f", title="Fuzzy F1"),
            ],
        )
        .properties(width=200, height=250, title="Overall Exact F1")
    )


plot_overall_f1(agg)


alt.Chart(...)

### Exact vs. fuzzy gap


In [8]:
def plot_exact_vs_fuzzy(df: pl.DataFrame) -> alt.LayerChart:
    domain = [0.6, 1.0]
    df_pd = df.with_columns(
        pl.concat_str(
            [pl.col("provider"), pl.lit(" "), pl.col("tier"), pl.lit(" "), pl.col("strategy")]
        ).alias("label")
    ).to_pandas()
    diagonal_data = pl.DataFrame({"x": [0.6, 1.0], "y": [0.6, 1.0]}).to_pandas()
    diagonal = (
        alt.Chart(diagonal_data)
        .mark_line(color="gray", strokeDash=[4, 4])
        .encode(
            x=alt.X("x:Q", scale=alt.Scale(domain=domain)),
            y=alt.Y("y:Q", scale=alt.Scale(domain=domain)),
        )
    )
    points = (
        alt.Chart(df_pd)
        .mark_point(filled=True, size=100)
        .encode(
            x=alt.X("exact_f1_mean:Q", title="Exact F1", scale=alt.Scale(domain=domain)),
            y=alt.Y("fuzzy_f1_mean:Q", title="Fuzzy F1", scale=alt.Scale(domain=domain)),
            color=alt.Color("provider:N", title="Provider"),
            shape=alt.Shape("strategy:N", title="Strategy"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("exact_f1_mean:Q", format=".3f", title="Exact F1"),
                alt.Tooltip("fuzzy_f1_mean:Q", format=".3f", title="Fuzzy F1"),
            ],
        )
    )
    return (diagonal + points).properties(width=300, height=300, title="Exact vs. Fuzzy F1")


plot_exact_vs_fuzzy(agg)


alt.LayerChart(...)

### Per-field breakdown


In [9]:
def plot_field_heatmap(df: pl.DataFrame) -> alt.Chart:
    field_cols = [
        "exact_effective_date_f1_mean",
        "exact_party_f1_mean",
        "exact_jurisdiction_f1_mean",
        "exact_term_f1_mean",
    ]
    melted = (
        df.with_columns(
            pl.concat_str(
                [pl.col("provider"), pl.lit(" "), pl.col("tier"), pl.lit(" "), pl.col("strategy")]
            ).alias("experiment")
        )
        .select(["experiment"] + field_cols)
        .unpivot(index="experiment", variable_name="column", value_name="f1")
        .with_columns(
            pl.col("column")
            .str.replace("exact_", "")
            .str.replace("_f1_mean", "")
            .alias("field")
        )
    )
    return (
        alt.Chart(melted.to_pandas())
        .mark_rect()
        .encode(
            x=alt.X(
                "field:N",
                title=None,
                sort=["effective_date", "party", "jurisdiction", "term"],
            ),
            y=alt.Y("experiment:N", title=None),
            color=alt.Color(
                "f1:Q",
                title="Exact F1",
                scale=alt.Scale(scheme="blues", domain=[0.5, 1.0]),
            ),
            tooltip=[
                alt.Tooltip("experiment:N"),
                alt.Tooltip("field:N"),
                alt.Tooltip("f1:Q", format=".3f", title="Exact F1"),
            ],
        )
        .properties(title="Per-field Exact F1", width=280, height=300)
    )


plot_field_heatmap(agg)


alt.Chart(...)

## Score distributions


In [10]:
def plot_f1_distributions(df: pl.DataFrame) -> alt.Chart:
    df_pd = (
        df.with_columns(
            pl.concat_str(
                [pl.col("provider"), pl.lit(" "), pl.col("tier"), pl.lit(" "), pl.col("strategy")]
            ).alias("experiment")
        )
        .to_pandas()
    )
    return (
        alt.Chart(df_pd)
        .mark_boxplot(extent="min-max", size=15)
        .encode(
            x=alt.X("exact_f1:Q", title="Exact F1", scale=alt.Scale(domain=[-0.05, 1.05])),
            y=alt.Y("experiment:N", title=None),
            color=alt.Color("provider:N", title="Provider"),
        )
        .properties(
            title="Per-document Exact F1 Distribution",
            width=400,
            height=340,
        )
    )


plot_f1_distributions(per_doc)


alt.Chart(...)

## Efficiency

### Cost vs. accuracy


In [11]:
def plot_cost_vs_f1(df: pl.DataFrame) -> alt.Chart:
    df_pd = df.with_columns(
        pl.concat_str(
            [pl.col("provider"), pl.lit(" "), pl.col("tier"), pl.lit(" "), pl.col("strategy")]
        ).alias("label")
    ).to_pandas()
    return (
        alt.Chart(df_pd)
        .mark_point(filled=True, size=100)
        .encode(
            x=alt.X(
                "cost_mean:Q",
                title="Mean cost per document (USD)",
                axis=alt.Axis(format="$.4f"),
            ),
            y=alt.Y(
                "exact_f1_mean:Q",
                title="Exact F1",
                scale=alt.Scale(domain=[0.6, 1.0]),
            ),
            color=alt.Color("provider:N", title="Provider"),
            shape=alt.Shape("tier:N", title="Tier"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("exact_f1_mean:Q", format=".3f", title="Exact F1"),
                alt.Tooltip("cost_mean:Q", format="$.5f", title="Cost/doc"),
                alt.Tooltip("cost_total:Q", format="$.3f", title="Total cost"),
            ],
        )
        .properties(title="Cost vs. Accuracy Trade-off", width=380, height=280)
    )


plot_cost_vs_f1(agg)


alt.Chart(...)

### Latency


In [12]:
def plot_latency(df: pl.DataFrame) -> alt.LayerChart:
    df_pd = df.with_columns(
        pl.concat_str(
            [pl.col("provider"), pl.lit(" "), pl.col("tier"), pl.lit(" "), pl.col("strategy")]
        ).alias("label")
    ).to_pandas()
    bars = (
        alt.Chart(df_pd)
        .mark_bar()
        .encode(
            x=alt.X("latency_mean:Q", title="Latency (s)"),
            y=alt.Y("label:N", sort="-x", title=None),
            color=alt.Color("provider:N", title="Provider"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("latency_mean:Q", format=".2f", title="Mean (s)"),
                alt.Tooltip("latency_median:Q", format=".2f", title="Median (s)"),
                alt.Tooltip("latency_p95:Q", format=".2f", title="P95 (s)"),
            ],
        )
    )
    p95 = (
        alt.Chart(df_pd)
        .mark_tick(color="black", thickness=2, size=15)
        .encode(
            x=alt.X("latency_p95:Q"),
            y=alt.Y("label:N", sort="-x"),
            tooltip=[alt.Tooltip("latency_p95:Q", format=".2f", title="P95 (s)")],
        )
    )
    return (bars + p95).properties(
        title="Mean Latency per Document (tick = P95)", width=380, height=320
    )


plot_latency(agg)


alt.LayerChart(...)

## Conclusions

- **Gemini dominates at standard tier**, achieving the highest exact F1 (~0.918) across both strategies, with negligible difference between agentic-multimodal and single-pass text. This suggests its vision capabilities are not strictly necessary for this task at standard scale.

- **Single-pass text is the best value at lite tier**. Gemini lite single-pass reaches 0.896 exact F1 at ~$0.001/doc and 1.4 s median latency — far outpacing any agentic configuration on cost and speed while remaining competitive on accuracy.

- **Agentic multimodal benefits more from a tier upgrade** than single-pass. Claude’s agentic exact F1 jumps +11 pp from lite to standard; GPT’s improves +22 pp. Single-pass gains only 1–3 pp, pointing to a performance ceiling at lite that the heavier approach breaks through at standard.

- **`jurisdiction` and `party` are the most reliably extracted fields** across all experiments. `term` consistently lags by 5–25 pp, reflecting its higher linguistic variability and frequent absence in documents.

- **Per-document score variance is high** (std 0.15–0.29), indicating that document difficulty is a stronger driver of per-document outcomes than model choice for borderline cases.
